# 03. Final model training and artifact export

## Цель

В этом notebook обучается и сохраняется финальный ансамбль для предсказания цены автомобиля.

Выбор архитектуры, параметров, весов ансамбля и conditional calibration был сделан в `02_experiments_model_selection.ipynb` на OOF- и nested-validation. Здесь **не выполняется новый подбор гиперпараметров**: утверждённые компоненты обучаются на всей обучающей выборке, после чего сохраняются все артефакты, необходимые для отдельного инференса.

## Результат

В папке `models/final_ensemble_12_66/` будут сохранены:

- Ridge и Text Ridge pipelines;
- CatBoost-модели V5, V8, Stats и V10;
- reference-таблица для retrieval;
- reference-данные для расчёта target statistics;
- схемы признаков и финальная конфигурация ансамбля;
- `requirements.txt` для фиксации окружения.

## 1. Импорты и пути проекта

Все пути определяются относительно корня проекта. Notebook ожидает, что уже были успешно выполнены:

1. `01_data_eda_features.ipynb` — создал V8-матрицы признаков;
2. `02_experiments_model_selection.ipynb` — сохранил зафиксированный рецепт финального ансамбля.

In [1]:
# ==============================================================
# 1. PROJECT SETUP
# ==============================================================

from pathlib import Path
import gc
import json
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# --------------------------------------------------------------
# Определяем корень проекта независимо от current working dir.
# --------------------------------------------------------------

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [current_path, *current_path.parents]
        if (candidate / "data").exists()
        and (candidate / "train_data").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не удалось определить корень проекта.\n"
        f"Текущая папка: {current_path}\n\n"
        "Ожидается структура:\n"
        "shift_ml/\n"
        "├── data/\n"
        "├── train_data/\n"
        "├── models/\n"
        "├── notebooks/\n"
        "└── reports/"
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
FINAL_MODEL_DIR = MODELS_DIR / "final_ensemble_12_66"

for directory in [
    REPORTS_DIR,
    MODELS_DIR,
    FINAL_MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COLUMN = "Цена"
ID_COLUMN = "car_id"

pd.set_option("display.max_columns", 100)

print("Project root:", PROJECT_ROOT)
print("Prepared data directory:", PROCESSED_DIR)
print("Final artifacts directory:", FINAL_MODEL_DIR)

Project root: C:\temp\shift_ml
Prepared data directory: C:\temp\shift_ml\data\processed
Final artifacts directory: C:\temp\shift_ml\models\final_ensemble_12_66


## 2. Загрузка замороженной конфигурации

Финальные веса ансамбля и параметры conditional calibration не переоптимизируются в этом notebook. Они были получены в исследовательском notebook по OOF-предсказаниям и nested validation, а затем подтверждены leaderboard-результатом `12.66`.

In [2]:
# ==============================================================
# 2. LOAD FROZEN FINAL RECIPE
# ==============================================================

FINAL_RECIPE_PATH = (
    REPORTS_DIR
    / "02_final_ensemble_recipe_12_66.json"
)

if not FINAL_RECIPE_PATH.exists():
    raise FileNotFoundError(
        "Не найден итоговый рецепт ансамбля:\n"
        f"{FINAL_RECIPE_PATH}\n\n"
        "Сначала полностью выполните "
        "02_experiments_model_selection.ipynb."
    )

with open(
    FINAL_RECIPE_PATH,
    mode="r",
    encoding="utf-8",
) as file:
    FINAL_ENSEMBLE_CONFIG = json.load(file)

expected_components = {
    "ridge",
    "v5",
    "v6",
    "v8",
    "stats",
    "text",
    "retrieval_k3_pred",
    "v10_stats_v8_pred",
}

found_components = set(
    FINAL_ENSEMBLE_CONFIG["ensemble_weights"]
)

if found_components != expected_components:
    raise ValueError(
        "Неожиданный набор компонентов в final recipe.\n"
        f"Ожидалось: {sorted(expected_components)}\n"
        f"Получено: {sorted(found_components)}"
    )

weights_sum = sum(
    FINAL_ENSEMBLE_CONFIG["ensemble_weights"].values()
)

if not np.isclose(weights_sum, 1.0, atol=1e-6):
    raise ValueError(
        "Веса ансамбля должны суммироваться в 1. "
        f"Текущая сумма: {weights_sum}"
    )

print(
    "Frozen leaderboard score:",
    FINAL_ENSEMBLE_CONFIG["leaderboard_score"],
)

display(
    pd.DataFrame(
        {
            "component": list(
                FINAL_ENSEMBLE_CONFIG[
                    "ensemble_weights"
                ].keys()
            ),
            "weight": list(
                FINAL_ENSEMBLE_CONFIG[
                    "ensemble_weights"
                ].values()
            ),
        }
    )
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)

Frozen leaderboard score: 12.66


,component,weight
0,v8,0.348682
1,ridge,0.196562
2,v10_stats_v8_pred,0.161980
3,retrieval_k3_pred,0.132080
4,stats,0.101857
5,text,0.038848
6,v5,0.019991
7,v6,0.000000


## 3. Загрузка подготовленных данных

Здесь загружаются только артефакты из Notebook 1. Feature engineering повторно не выполняется: это исключает расхождение между notebook'ами и фиксирует один источник признаков.

Canonical-таблицы нужны только для двух текстовых источников:

- Text Ridge использует исходные текстовые поля;
- V5 получает исходное полное название как дополнительную категориальную информацию.

In [3]:
# ==============================================================
# 3. LOAD PREPARED DATA
# ==============================================================

DATA_PATHS = {
    "train_model_input": (
        PROCESSED_DIR / "train_model_input_v8.parquet"
    ),
    "test_model_input": (
        PROCESSED_DIR / "test_model_input_v8.parquet"
    ),
    "X_train_v8": (
        PROCESSED_DIR / "X_train_v8.parquet"
    ),
    "X_test_v8": (
        PROCESSED_DIR / "X_test_v8.parquet"
    ),
    "feature_schema": (
        PROCESSED_DIR / "feature_schema.json"
    ),
    "train_canonical": (
        PROCESSED_DIR / "train_canonical.parquet"
    ),
    "test_canonical": (
        PROCESSED_DIR / "X_test_canonical.parquet"
    ),
}

missing_data_files = {
    name: path
    for name, path in DATA_PATHS.items()
    if not path.exists()
}

if missing_data_files:
    formatted = "\n".join(
        f"— {name}: {path}"
        for name, path in missing_data_files.items()
    )

    raise FileNotFoundError(
        "Не найдены входные артефакты.\n\n"
        "Сначала полностью выполните "
        "01_data_eda_features.ipynb.\n\n"
        f"Не найдены:\n{formatted}"
    )

train_model_input = pd.read_parquet(
    DATA_PATHS["train_model_input"]
)

test_model_input = pd.read_parquet(
    DATA_PATHS["test_model_input"]
)

X_train_v8 = pd.read_parquet(
    DATA_PATHS["X_train_v8"]
)

X_test_v8 = pd.read_parquet(
    DATA_PATHS["X_test_v8"]
)

train_canonical = pd.read_parquet(
    DATA_PATHS["train_canonical"]
)

test_canonical = pd.read_parquet(
    DATA_PATHS["test_canonical"]
)

with open(
    DATA_PATHS["feature_schema"],
    mode="r",
    encoding="utf-8",
) as file:
    feature_schema = json.load(file)

y_train = train_model_input[TARGET_COLUMN].astype(float)
train_ids = train_model_input[ID_COLUMN].astype(str)
test_ids = test_model_input[ID_COLUMN].astype(str)

assert len(train_model_input) == 8340
assert len(test_model_input) == 8341
assert len(X_train_v8) == len(train_model_input)
assert len(X_test_v8) == len(test_model_input)
assert X_train_v8.columns.tolist() == X_test_v8.columns.tolist()
assert X_train_v8.columns.tolist() == feature_schema["feature_columns"]
assert y_train.notna().all()
assert train_ids.nunique() == len(train_ids)
assert test_ids.nunique() == len(test_ids)

print("Train model input:", train_model_input.shape)
print("Test model input:", test_model_input.shape)
print("X_train_v8:", X_train_v8.shape)
print("X_test_v8:", X_test_v8.shape)

display(y_train.describe().to_frame("Цена"))

Train model input: (8340, 75)
Test model input: (8341, 74)
X_train_v8: (8340, 73)
X_test_v8: (8341, 73)


,Цена
count,8.340000e+03
mean,3.712697e+04
std,3.679723e+04
min,9.000000e+02
25%,1.944075e+04
50%,2.962000e+04
75%,4.399000e+04
max,1.500000e+06


## 4. Общие функции подготовки представлений

Разные модели используют разные представления одних и тех же данных:

- CatBoost получает числовые и категориальные признаки напрямую;
- Ridge получает numeric + one-hot encoding;
- Text Ridge работает со строковым описанием;
- retrieval использует только характеристики, важные для поиска ближайших аналогов.

Перед обучением категориальные поля приводятся к строке с единым значением `__MISSING__` для пропусков.

In [4]:
# ==============================================================
# 4. COMMON PREPARATION HELPERS
# ==============================================================

categorical_v8 = feature_schema["categorical_columns"]
numeric_v8 = feature_schema["numeric_columns"]

missing_categorical = [
    column
    for column in categorical_v8
    if column not in X_train_v8.columns
]

missing_numeric = [
    column
    for column in numeric_v8
    if column not in X_train_v8.columns
]

if missing_categorical or missing_numeric:
    raise KeyError(
        "Схема признаков не совпадает с X_train_v8.\n"
        f"Missing categorical: {missing_categorical}\n"
        f"Missing numeric: {missing_numeric}"
    )


def prepare_catboost_frame(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    """Приводит категории к единому формату, ожидаемому CatBoost."""
    result = frame.copy()

    for column in categorical_columns:
        result[column] = (
            result[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return result


def align_canonical_by_id(
    canonical_frame: pd.DataFrame,
    ids: pd.Series,
) -> pd.DataFrame:
    """Возвращает canonical-строки в порядке id подготовленной матрицы."""
    result = canonical_frame.copy()
    result[ID_COLUMN] = result[ID_COLUMN].astype(str)
    result = result.set_index(ID_COLUMN, drop=False)

    missing_ids = set(ids.astype(str)) - set(result.index)

    if missing_ids:
        raise KeyError(
            "Не все car_id найдены в canonical-таблице. "
            f"Пример: {list(missing_ids)[:5]}"
        )

    return result.loc[ids.astype(str)].reset_index(drop=True)


X_train_cb = prepare_catboost_frame(
    X_train_v8,
    categorical_v8,
)

X_test_cb = prepare_catboost_frame(
    X_test_v8,
    categorical_v8,
)

train_canonical_aligned = align_canonical_by_id(
    train_canonical,
    train_ids,
)

test_canonical_aligned = align_canonical_by_id(
    test_canonical,
    test_ids,
)

assert X_train_cb.columns.tolist() == X_test_cb.columns.tolist()
assert len(train_canonical_aligned) == len(X_train_cb)
assert len(test_canonical_aligned) == len(X_test_cb)

print("CatBoost V8 features:", X_train_cb.shape[1])
print("Categorical V8 features:", len(categorical_v8))
print("Numeric V8 features:", len(numeric_v8))

CatBoost V8 features: 73
Categorical V8 features: 22
Numeric V8 features: 51


## 5. Ridge на табличных признаках

Ridge — линейный компонент ансамбля. Он обучается на `log1p(Цена)`, что уменьшает влияние дорогих автомобилей и лучше соответствует относительной природе MAPE.

Хотя Ridge слабее CatBoost как отдельная модель, он сохраняется в ансамбле из-за иной структуры ошибок: линейная модель лучше обобщает часть глобальных зависимостей и уменьшает риск того, что все компоненты повторяют одну и ту же ошибку.

In [5]:
# ==============================================================
# 5. TRAIN AND SAVE RIDGE
# ==============================================================

def make_one_hot_encoder() -> OneHotEncoder:
    """Поддержка нескольких версий scikit-learn."""
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


ridge_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="median"),
                    ),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_v8,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        ),
                    ),
                    (
                        "one_hot",
                        make_one_hot_encoder(),
                    ),
                ]
            ),
            categorical_v8,
        ),
    ],
    remainder="drop",
)

ridge_final_model = Pipeline(
    steps=[
        ("preprocessor", ridge_preprocessor),
        (
            "model",
            Ridge(alpha=0.1, solver="lsqr"),
        ),
    ]
)

ridge_final_model.fit(
    X_train_cb,
    np.log1p(y_train),
)

RIDGE_MODEL_PATH = (
    FINAL_MODEL_DIR
    / "ridge_log_target_final.joblib"
)

joblib.dump(
    ridge_final_model,
    RIDGE_MODEL_PATH,
)

print("Saved:", RIDGE_MODEL_PATH)

Saved: C:\temp\shift_ml\models\final_ensemble_12_66\ridge_log_target_final.joblib


## 6. Text Ridge

Text Ridge использует отдельное текстовое представление автомобиля. В него входят марка, модель, полное название, двигатель, привод, коробка и тип кузова.

TF-IDF выделяет слова и пары слов — например, обозначения комплектаций, двигателей и специальных версий. Такой компонент полезен, потому что часть редких текстовых сигналов может быть представлена в структурных признаках неполно.

In [6]:
# ==============================================================
# 6. TRAIN AND SAVE TEXT RIDGE
# ==============================================================

TEXT_COLUMNS = [
    "Бренд",
    "Модель",
    "Полное название",
    "Двигатель",
    "Привод",
    "КПП",
    "Тип кузова",
]

missing_text_columns = [
    column
    for column in TEXT_COLUMNS
    if column not in train_canonical_aligned.columns
]

if missing_text_columns:
    raise KeyError(
        "В canonical-данных отсутствуют текстовые поля: "
        f"{missing_text_columns}"
    )


def make_vehicle_text(
    frame: pd.DataFrame,
) -> pd.Series:
    """Формирует единый текстовый документ на один автомобиль."""
    text_parts = [
        frame[column]
        .astype("string")
        .fillna("")
        .str.strip()
        for column in TEXT_COLUMNS
    ]

    return pd.concat(text_parts, axis=1).agg(
        " ".join,
        axis=1,
    )


train_vehicle_text = make_vehicle_text(
    train_canonical_aligned
)

text_ridge_final_model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_features=50_000,
                sublinear_tf=True,
            ),
        ),
        (
            "model",
            Ridge(alpha=1.0, solver="lsqr"),
        ),
    ]
)

text_ridge_final_model.fit(
    train_vehicle_text,
    np.log1p(y_train),
)

TEXT_MODEL_PATH = (
    FINAL_MODEL_DIR
    / "text_tfidf_ridge_final.joblib"
)

joblib.dump(
    text_ridge_final_model,
    TEXT_MODEL_PATH,
)

print("Saved:", TEXT_MODEL_PATH)
print("Text vocabulary size:", len(
    text_ridge_final_model.named_steps[
        "tfidf"
    ].vocabulary_
))

Saved: C:\temp\shift_ml\models\final_ensemble_12_66\text_tfidf_ridge_final.joblib
Text vocabulary size: 6053


## 7. Target statistics без утечки

Цена автомобиля зависит от группы: марки, модели, года выпуска и версии. Чтобы передать модели эту рыночную структуру, создаются сглаженные статистики логарифма цены для шести групп.

Для train применяется cross-fitting: каждая строка получает статистику, рассчитанную без собственного таргета. Это критично: расчёт групповой средней с использованием той же строки был бы target leakage.

В дальнейшем для новых объектов используются statistics, рассчитанные по всей обучающей выборке; исходные reference-данные сохраняются отдельно.

In [7]:
# ==============================================================
# 7. TARGET STATISTICS HELPERS
# ==============================================================

GROUP_SPECS_V1 = {
    "brand_model": ["Бренд", "Модель"],
    "brand_model_year": [
        "Бренд",
        "Модель",
        "Год выпуска",
    ],
    "title_prefix3": [
        "Название_префикс_3",
    ],
    "title_prefix3_year": [
        "Название_префикс_3",
        "Год выпуска",
    ],
    "title_normalized": [
        "Название_нормализованное_без_года",
    ],
    "title_normalized_year": [
        "Название_нормализованное_без_года",
        "Год выпуска",
    ],
}

TARGET_STATS_SMOOTHING = 20.0
TARGET_STATS_N_SPLITS = 5

target_stats_group_columns = list(
    dict.fromkeys(
        column
        for columns in GROUP_SPECS_V1.values()
        for column in columns
    )
)

missing_group_columns = [
    column
    for column in target_stats_group_columns
    if column not in X_train_cb.columns
]

if missing_group_columns:
    raise KeyError(
        "В V8 feature set не хватает колонок для target stats: "
        f"{missing_group_columns}"
    )


def make_group_key(
    frame: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    """Создаёт устойчивый составной ключ для статистической группы."""
    key = None

    for column in columns:
        series = frame[column]

        if pd.api.types.is_numeric_dtype(series):
            part = (
                pd.to_numeric(
                    series,
                    errors="coerce",
                )
                .round(4)
                .astype("Float64")
                .astype("string")
            )
        else:
            part = series.astype("string")

        part = (
            part.fillna("__MISSING__")
            .str.strip()
            .str.upper()
        )

        key = (
            part
            if key is None
            else key.str.cat(part, sep="|||")
        )

    return key


def build_target_stats(
    X_reference: pd.DataFrame,
    y_reference: pd.Series,
    X_apply: pd.DataFrame,
    group_specs: dict[str, list[str]],
    smoothing: float,
) -> pd.DataFrame:
    """Строит count и сглаженное среднее log-price на reference."""
    X_reference = X_reference.reset_index(drop=True)
    X_apply = X_apply.reset_index(drop=True)

    target_log = np.log1p(
        pd.Series(y_reference)
        .reset_index(drop=True)
        .astype(float)
    )

    global_mean = float(target_log.mean())
    output = pd.DataFrame(index=X_apply.index)

    for group_name, group_columns in group_specs.items():
        reference_key = make_group_key(
            X_reference,
            group_columns,
        )

        apply_key = make_group_key(
            X_apply,
            group_columns,
        )

        group_table = pd.DataFrame(
            {
                "key": reference_key.to_numpy(),
                "target_log": target_log.to_numpy(),
            }
        )

        grouped = (
            group_table
            .groupby("key", sort=False)
            .agg(
                count=("target_log", "size"),
                target_sum=("target_log", "sum"),
            )
        )

        grouped["smooth_mean"] = (
            grouped["target_sum"]
            + smoothing * global_mean
        ) / (
            grouped["count"] + smoothing
        )

        counts = (
            apply_key
            .map(grouped["count"])
            .fillna(0)
            .to_numpy(dtype=float)
        )

        smoothed_means = (
            apply_key
            .map(grouped["smooth_mean"])
            .fillna(global_mean)
            .to_numpy(dtype=float)
        )

        output[
            f"{group_name}__count"
        ] = np.log1p(counts)

        output[
            f"{group_name}__log_mean"
        ] = smoothed_means

    return output


def make_stratification_bins(
    y: pd.Series,
) -> np.ndarray:
    """Делит таргет на квантили для устойчивого внутреннего CV."""
    return np.asarray(
        pd.qcut(
            y,
            q=10,
            labels=False,
            duplicates="drop",
        )
    )


def make_cross_fitted_target_stats(
    X_fit: pd.DataFrame,
    y_fit: pd.Series,
    X_apply: pd.DataFrame,
    group_specs: dict[str, list[str]],
    smoothing: float = TARGET_STATS_SMOOTHING,
    n_splits: int = TARGET_STATS_N_SPLITS,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Возвращает:
    - cross-fitted statistics для X_fit;
    - statistics по full X_fit для X_apply.
    """
    X_fit = X_fit.reset_index(drop=True)
    X_apply = X_apply.reset_index(drop=True)
    y_fit = pd.Series(y_fit).reset_index(drop=True)

    expected_columns = [
        output_column
        for group_name in group_specs
        for output_column in [
            f"{group_name}__count",
            f"{group_name}__log_mean",
        ]
    ]

    fitted_stats = pd.DataFrame(
        index=X_fit.index,
        columns=expected_columns,
        dtype=float,
    )

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    stratification_bins = make_stratification_bins(y_fit)

    for fit_index, valid_index in inner_cv.split(
        X_fit,
        stratification_bins,
    ):
        fold_stats = build_target_stats(
            X_reference=X_fit.iloc[fit_index],
            y_reference=y_fit.iloc[fit_index],
            X_apply=X_fit.iloc[valid_index],
            group_specs=group_specs,
            smoothing=smoothing,
        )

        fitted_stats.iloc[valid_index] = (
            fold_stats.to_numpy()
        )

    apply_stats = build_target_stats(
        X_reference=X_fit,
        y_reference=y_fit,
        X_apply=X_apply,
        group_specs=group_specs,
        smoothing=smoothing,
    )

    if fitted_stats.isna().any().any():
        raise ValueError(
            "Cross-fitted target statistics содержат NaN."
        )

    return fitted_stats, apply_stats

## 8. Подготовка финальных матриц V5, V8, Stats и V10

- **V8** — основная нелинейная CatBoost-модель на всех финальных структурных признаках.
- **V5** — дополнительная title-heavy версия: использует V8-признаки и исходные `Полное название`/`Цвет`. Её вес небольшой, но она добавляет иную структуру ошибок.
- **Stats** — более компактная версия с target statistics и без расширенных V8 title-маркеров.
- **V10** — V8 плюс fold-safe target statistics.

Так модели намеренно не полностью совпадают: ансамблю важна не только сила отдельных компонентов, но и разнообразие их ошибок.

In [8]:
# ==============================================================
# 8. BUILD FINAL MODEL MATRICES
# ==============================================================

# ----- V5: V8 features + raw full title and colour -----

V5_ADDITIONAL_RAW_COLUMNS = [
    "Полное название",
    "Цвет",
]

missing_v5_raw_columns = [
    column
    for column in V5_ADDITIONAL_RAW_COLUMNS
    if column not in train_canonical_aligned.columns
]

if missing_v5_raw_columns:
    raise KeyError(
        "Для V5 не хватает raw-полей: "
        f"{missing_v5_raw_columns}"
    )

X_train_v5 = X_train_cb.copy()
X_test_v5 = X_test_cb.copy()

for column in V5_ADDITIONAL_RAW_COLUMNS:
    X_train_v5[column] = (
        train_canonical_aligned[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_v5[column] = (
        test_canonical_aligned[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

categorical_v5 = [
    *categorical_v8,
    *V5_ADDITIONAL_RAW_COLUMNS,
]


# ----- Stats: compact base + 12 cross-fitted target statistics -----

V8_SUPPLEMENTARY_TITLE_FEATURES = [
    "Название_комплектация_1",
    "Название_комплектация_2",
    "Название_комплектация_3",
    "Название_первые_2",
    "Название_первые_3",
    "Название_число_токенов",
    "Название_флаг_QUATTRO",
    "Название_флаг_XDRIVE",
    "Название_флаг_4MATIC",
    "Название_флаг_ALLGRIP",
    "Название_флаг_ALLTRACK",
    "Название_флаг_EQUIPPED",
]

stats_base_columns = [
    column
    for column in X_train_cb.columns
    if column not in V8_SUPPLEMENTARY_TITLE_FEATURES
]

X_train_stats_base = X_train_cb[
    stats_base_columns
].copy()

X_test_stats_base = X_test_cb[
    stats_base_columns
].copy()

categorical_stats = [
    column
    for column in categorical_v8
    if column in stats_base_columns
]


# ----- Cross-fitted target statistics for full training -----

X_train_target_stats, X_test_target_stats = (
    make_cross_fitted_target_stats(
        X_fit=X_train_cb,
        y_fit=y_train,
        X_apply=X_test_cb,
        group_specs=GROUP_SPECS_V1,
        smoothing=TARGET_STATS_SMOOTHING,
        n_splits=TARGET_STATS_N_SPLITS,
        random_state=RANDOM_STATE,
    )
)

X_train_stats = pd.concat(
    [
        X_train_stats_base.reset_index(drop=True),
        X_train_target_stats.reset_index(drop=True),
    ],
    axis=1,
)

X_test_stats = pd.concat(
    [
        X_test_stats_base.reset_index(drop=True),
        X_test_target_stats.reset_index(drop=True),
    ],
    axis=1,
)

X_train_v10 = pd.concat(
    [
        X_train_cb.reset_index(drop=True),
        X_train_target_stats.reset_index(drop=True),
    ],
    axis=1,
)

X_test_v10 = pd.concat(
    [
        X_test_cb.reset_index(drop=True),
        X_test_target_stats.reset_index(drop=True),
    ],
    axis=1,
)

assert X_train_v5.columns.tolist() == X_test_v5.columns.tolist()
assert X_train_stats.columns.tolist() == X_test_stats.columns.tolist()
assert X_train_v10.columns.tolist() == X_test_v10.columns.tolist()

assert len(X_train_v5) == len(y_train)
assert len(X_train_stats) == len(y_train)
assert len(X_train_v10) == len(y_train)

print("V5 features:", X_train_v5.shape)
print("V8 features:", X_train_cb.shape)
print("Stats features:", X_train_stats.shape)
print("V10 features:", X_train_v10.shape)

display(
    X_train_target_stats.head(3)
)

V5 features: (8340, 75)
V8 features: (8340, 73)
Stats features: (8340, 73)
V10 features: (8340, 85)


,brand_model__count,brand_model__log_mean,brand_model_year__count,brand_model_year__log_mean,title_prefix3__count,title_prefix3__log_mean,title_prefix3_year__count,title_prefix3_year__log_mean,title_normalized__count,title_normalized__log_mean,title_normalized_year__count,title_normalized_year__log_mean
0,3.135494,10.474414,0.693147,10.295563,1.791759,10.408583,0.000000,10.280514,0.000000,10.280514,0.000000,10.280514
1,2.197225,10.382854,1.945910,10.434298,0.000000,10.279960,0.000000,10.279960,0.000000,10.279960,0.000000,10.279960
2,4.770685,10.132591,1.098612,10.207400,4.770685,10.132591,1.098612,10.207400,2.197225,10.077536,0.693147,10.235925


## 9. Обучение CatBoost-моделей

Все CatBoost-модели обучаются на `log1p(Цена)` с фиксированными гиперпараметрами. В финальном обучении используется вся доступная обучающая выборка, поэтому `eval_set` и early stopping не применяются: число итераций фиксировано заранее по результатам исследований.

In [9]:
# ==============================================================
# 9. TRAIN AND SAVE CATBOOST MODELS
# ==============================================================

CATBOOST_PARAMS = {
    "loss_function": "RMSE",
    "iterations": 3000,
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 5,
    "random_seed": RANDOM_STATE,
    "verbose": 500,
    "allow_writing_files": False,
}

log_y_train = np.log1p(y_train)


def train_and_save_catboost(
    model_name: str,
    X_train: pd.DataFrame,
    categorical_columns: list[str],
    output_filename: str,
) -> tuple[CatBoostRegressor, Path]:
    """Обучает CatBoost на полном train и сохраняет .cbm."""
    model = CatBoostRegressor(**CATBOOST_PARAMS)

    model.fit(
        X_train,
        log_y_train,
        cat_features=categorical_columns,
    )

    output_path = FINAL_MODEL_DIR / output_filename
    model.save_model(output_path)

    print(
        f"Saved {model_name}: {output_path.name} | "
        f"features: {X_train.shape[1]}"
    )

    return model, output_path


v5_final_model, V5_MODEL_PATH = train_and_save_catboost(
    model_name="V5 title-heavy",
    X_train=X_train_v5,
    categorical_columns=categorical_v5,
    output_filename="catboost_v5_title_final.cbm",
)

gc.collect()

v8_final_model, V8_MODEL_PATH = train_and_save_catboost(
    model_name="V8",
    X_train=X_train_cb,
    categorical_columns=categorical_v8,
    output_filename="catboost_v8_final.cbm",
)

gc.collect()

stats_final_model, STATS_MODEL_PATH = train_and_save_catboost(
    model_name="Stats",
    X_train=X_train_stats,
    categorical_columns=categorical_stats,
    output_filename="catboost_stats_final.cbm",
)

gc.collect()

v10_final_model, V10_MODEL_PATH = train_and_save_catboost(
    model_name="V10",
    X_train=X_train_v10,
    categorical_columns=categorical_v8,
    output_filename="catboost_v10_final.cbm",
)

gc.collect()

0:	learn: 0.6516796	total: 274ms	remaining: 13m 42s
500:	learn: 0.1523942	total: 59.4s	remaining: 4m 56s
1000:	learn: 0.1181186	total: 2m 3s	remaining: 4m 5s
1500:	learn: 0.0968761	total: 3m 6s	remaining: 3m 6s
2000:	learn: 0.0810859	total: 4m 10s	remaining: 2m 4s
2500:	learn: 0.0690596	total: 5m 12s	remaining: 1m 2s
2999:	learn: 0.0593364	total: 6m 15s	remaining: 0us
Saved V5 title-heavy: catboost_v5_title_final.cbm | features: 75
0:	learn: 0.6520204	total: 102ms	remaining: 5m 5s
500:	learn: 0.1536963	total: 54.9s	remaining: 4m 33s
1000:	learn: 0.1167246	total: 1m 55s	remaining: 3m 49s
1500:	learn: 0.0953792	total: 2m 54s	remaining: 2m 54s
2000:	learn: 0.0807595	total: 3m 57s	remaining: 1m 58s
2500:	learn: 0.0690728	total: 5m 9s	remaining: 1m 1s
2999:	learn: 0.0595741	total: 6m 27s	remaining: 0us
Saved V8: catboost_v8_final.cbm | features: 73
0:	learn: 0.6511618	total: 112ms	remaining: 5m 34s
500:	learn: 0.1418172	total: 1m 1s	remaining: 5m 5s
1000:	learn: 0.1066649	total: 2m 15s	rema

0

## 10. Reference-таблица для retrieval

Retrieval не является обучаемой параметрической моделью. Для прогнозирования он ищет ближайшие автомобили среди train-объектов, поэтому необходимо сохранить не только параметры расстояния, но и reference-таблицу с признаками и логарифмом цены.

Поиск аналогов устроен иерархически:

1. кандидаты среди той же пары `Бренд + Модель`;
2. fallback среди того же бренда;
3. fallback по всему train.

Цена строится как взвешенное среднее логарифмов цен трёх ближайших аналогов.

In [12]:
# ==============================================================
# 10. BUILD AND SAVE RETRIEVAL REFERENCE
# ==============================================================

RETRIEVAL_NUMERIC_SPECS = {
    "Год выпуска": 3.0,
    "Пробег_log": 0.70,
    "Двигатель_объём_л": 0.70,
    "Двигатель_цилиндры": 1.5,
    "Двери_число": 2.0,
    "Кресла_число": 2.0,
}

RETRIEVAL_CATEGORICAL_WEIGHTS = {
    "Топливо": 0.35,
    "КПП": 0.25,
    "Привод": 0.25,
    "Тип кузова": 0.30,
    "Штат": 0.10,
    "Название_комплектация_1": 0.35,
    "Название_комплектация_2": 0.15,
}

RETRIEVAL_REQUIRED_COLUMNS = list(
    dict.fromkeys(
        [
            "Бренд",
            "Модель",
            "Год выпуска",
            "Пробег_число",
            "Двигатель_объём_л",
            "Двигатель_цилиндры",
            "Двери_число",
            "Кресла_число",
            *RETRIEVAL_CATEGORICAL_WEIGHTS.keys(),
        ]
    )
)

missing_retrieval_columns = [
    column
    for column in RETRIEVAL_REQUIRED_COLUMNS
    if column not in X_train_cb.columns
]

if missing_retrieval_columns:
    raise KeyError(
        "Для retrieval не хватает признаков: "
        f"{missing_retrieval_columns}"
    )


def clean_retrieval_category(
    series: pd.Series,
) -> pd.Series:
    return (
        series.astype("string")
        .fillna("__MISSING__")
        .str.strip()
        .str.upper()
        .astype(str)
    )


def prepare_retrieval_reference(
    X: pd.DataFrame,
    target: pd.Series,
) -> pd.DataFrame:
    """Подготавливает и сохраняет train-объекты как базу аналогов."""
    result = X[
        RETRIEVAL_REQUIRED_COLUMNS
    ].copy().reset_index(drop=True)

    for column in [
        "Год выпуска",
        "Пробег_число",
        "Двигатель_объём_л",
        "Двигатель_цилиндры",
        "Двери_число",
        "Кресла_число",
    ]:
        result[column] = pd.to_numeric(
            result[column],
            errors="coerce",
        ).astype(float)

    result["Пробег_log"] = np.log1p(
        result["Пробег_число"]
    )

    for column in RETRIEVAL_CATEGORICAL_WEIGHTS:
        result[column] = clean_retrieval_category(
            result[column]
        )

    result["Бренд"] = clean_retrieval_category(
        result["Бренд"]
    )

    result["Модель"] = clean_retrieval_category(
        result["Модель"]
    )

    result["brand_model_key"] = (
        result["Бренд"]
        + "|||"
        + result["Модель"]
    )

    result["brand_key"] = result["Бренд"]

    result["target_log_price"] = np.log1p(
        pd.Series(target)
        .reset_index(drop=True)
        .astype(float)
    )

    return result


retrieval_reference = prepare_retrieval_reference(
    X_train_cb,
    y_train,
)

RETRIEVAL_REFERENCE_PATH = (
    FINAL_MODEL_DIR
    / "retrieval_reference.parquet"
)

retrieval_reference.to_parquet(
    RETRIEVAL_REFERENCE_PATH,
    index=False,
)

print("Saved:", RETRIEVAL_REFERENCE_PATH)
print("Reference shape:", retrieval_reference.shape)
display(retrieval_reference.head(3))

Saved: C:\temp\shift_ml\models\final_ensemble_12_66\retrieval_reference.parquet
Reference shape: (8340, 19)


,Бренд,Модель,Год выпуска,Пробег_число,Двигатель_объём_л,Двигатель_цилиндры,Двери_число,Кресла_число,Топливо,КПП,Привод,Тип кузова,Штат,Название_комплектация_1,Название_комплектация_2,Пробег_log,brand_model_key,brand_key,target_log_price
0,SUBARU,WRX,2018.0,29030.0,2.5,4.0,4.0,5.0,PREMIUM,MANUAL,AWD,SEDAN,VIC,STI,STI SPEC,10.276120,SUBARU|||WRX,SUBARU,11.034099
1,SSANGYONG,REXTON,2023.0,7.0,2.2,4.0,4.0,7.0,DIESEL,AUTOMATIC,AWD,SUV,NSW,ELX,ELX AWD,2.079442,SSANGYONG|||REXTON,SSANGYONG,10.799392
2,NISSAN,X-TRAIL,2011.0,248318.0,2.5,4.0,4.0,5.0,UNLEADED,MANUAL,4WD,SUV,ACT,ST,ST 4X4,12.422469,NISSAN|||X-TRAIL,NISSAN,9.053687


## 11. Экспорт target-statistics reference и схем моделей

Для будущего инференса недостаточно сохранить только CatBoost `.cbm`:

- V10 и Stats требуют групповые statistics, построенные по train;
- каждой модели необходим точный порядок колонок;
- CatBoost должен знать, какие поля категориальные;
- ensemble требует веса и calibration-параметры.

Поэтому ниже сохраняется полный контракт между обучением и инференсом.

In [13]:
# ==============================================================
# 11. SAVE MODEL SCHEMAS AND REFERENCE ARTIFACTS
# ==============================================================

# Для расчёта target statistics на новых объектах достаточно
# шести группирующих полей и log-price обучающих объектов.
target_stats_reference = X_train_cb[
    target_stats_group_columns
].copy()

target_stats_reference["target_log_price"] = np.log1p(
    y_train.to_numpy(dtype=float)
)

TARGET_STATS_REFERENCE_PATH = (
    FINAL_MODEL_DIR
    / "target_stats_reference.parquet"
)

target_stats_reference.to_parquet(
    TARGET_STATS_REFERENCE_PATH,
    index=False,
)

model_feature_schemas = {
    "ridge": {
        "feature_columns": X_train_cb.columns.tolist(),
        "categorical_columns": categorical_v8,
        "numeric_columns": numeric_v8,
    },
    "text_ridge": {
        "text_columns": TEXT_COLUMNS,
    },
    "v5": {
        "feature_columns": X_train_v5.columns.tolist(),
        "categorical_columns": categorical_v5,
    },
    "v8": {
        "feature_columns": X_train_cb.columns.tolist(),
        "categorical_columns": categorical_v8,
    },
    "stats": {
        "feature_columns": X_train_stats.columns.tolist(),
        "categorical_columns": categorical_stats,
        "base_feature_columns": stats_base_columns,
        "target_stats_columns": (
            X_train_target_stats.columns.tolist()
        ),
    },
    "v10": {
        "feature_columns": X_train_v10.columns.tolist(),
        "categorical_columns": categorical_v8,
        "base_feature_columns": X_train_cb.columns.tolist(),
        "target_stats_columns": (
            X_train_target_stats.columns.tolist()
        ),
    },
    "target_statistics": {
        "group_specs": GROUP_SPECS_V1,
        "smoothing": TARGET_STATS_SMOOTHING,
        "reference_columns": target_stats_group_columns,
    },
    "retrieval": {
        "required_columns": RETRIEVAL_REQUIRED_COLUMNS,
        "numeric_scales": RETRIEVAL_NUMERIC_SPECS,
        "categorical_weights": (
            RETRIEVAL_CATEGORICAL_WEIGHTS
        ),
        "k_neighbors": 3,
        "distance_epsilon": 0.15,
        "missing_numeric_penalty": 0.20,
        "missing_categorical_penalty": 0.15,
    },
}

MODEL_SCHEMAS_PATH = (
    FINAL_MODEL_DIR
    / "model_feature_schemas.json"
)

with open(
    MODEL_SCHEMAS_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        model_feature_schemas,
        file,
        ensure_ascii=False,
        indent=4,
    )

# Копия frozen recipe внутри той же папки, что и модели.
FINAL_CONFIG_PATH = (
    FINAL_MODEL_DIR
    / "ensemble_config.json"
)

with open(
    FINAL_CONFIG_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        FINAL_ENSEMBLE_CONFIG,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("Saved:", TARGET_STATS_REFERENCE_PATH)
print("Saved:", MODEL_SCHEMAS_PATH)
print("Saved:", FINAL_CONFIG_PATH)

Saved: C:\temp\shift_ml\models\final_ensemble_12_66\target_stats_reference.parquet
Saved: C:\temp\shift_ml\models\final_ensemble_12_66\model_feature_schemas.json
Saved: C:\temp\shift_ml\models\final_ensemble_12_66\ensemble_config.json


## 12. Контроль сохранённых артефактов

Проверяем, что каждая часть финального ансамбля сохранена. На этом этапе submission не строится: инференс будет вынесен в отдельный технический notebook или скрипт, который загрузит именно эти файлы.

In [14]:
# ==============================================================
# 12. ARTIFACT VALIDATION AND TRAINING MANIFEST
# ==============================================================

artifact_paths = {
    "Ridge pipeline": RIDGE_MODEL_PATH,
    "Text Ridge pipeline": TEXT_MODEL_PATH,
    "CatBoost V5": V5_MODEL_PATH,
    "CatBoost V8": V8_MODEL_PATH,
    "CatBoost Stats": STATS_MODEL_PATH,
    "CatBoost V10": V10_MODEL_PATH,
    "Retrieval reference": RETRIEVAL_REFERENCE_PATH,
    "Target statistics reference": (
        TARGET_STATS_REFERENCE_PATH
    ),
    "Model schemas": MODEL_SCHEMAS_PATH,
    "Ensemble config": FINAL_CONFIG_PATH,
}

missing_artifacts = [
    name
    for name, path in artifact_paths.items()
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Не удалось сохранить все финальные артефакты: "
        f"{missing_artifacts}"
    )

training_manifest = {
    "experiment_name": "final_ensemble_12_66",
    "leaderboard_score": FINAL_ENSEMBLE_CONFIG[
        "leaderboard_score"
    ],
    "train_rows": int(len(X_train_cb)),
    "test_rows": int(len(X_test_cb)),
    "v8_feature_count": int(X_train_cb.shape[1]),
    "v5_feature_count": int(X_train_v5.shape[1]),
    "stats_feature_count": int(X_train_stats.shape[1]),
    "v10_feature_count": int(X_train_v10.shape[1]),
    "artifacts": {
        name: str(path.relative_to(PROJECT_ROOT))
        for name, path in artifact_paths.items()
    },
}

TRAINING_MANIFEST_PATH = (
    FINAL_MODEL_DIR
    / "training_manifest.json"
)

with open(
    TRAINING_MANIFEST_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        training_manifest,
        file,
        ensure_ascii=False,
        indent=4,
    )

artifact_table = pd.DataFrame(
    {
        "artifact": list(artifact_paths.keys()),
        "path": [
            str(path)
            for path in artifact_paths.values()
        ],
        "size_kb": [
            round(path.stat().st_size / 1024, 2)
            for path in artifact_paths.values()
        ],
    }
)

print("Saved:", TRAINING_MANIFEST_PATH)
print("\n=== FINAL ARTIFACTS ===")
display(artifact_table)

Saved: C:\temp\shift_ml\models\final_ensemble_12_66\training_manifest.json

=== FINAL ARTIFACTS ===


,artifact,path,size_kb
0,Ridge pipeline,C:\temp\shift_ml\models\final_ensemble_12_66\r...,551.25
1,Text Ridge pipeline,C:\temp\shift_ml\models\final_ensemble_12_66\t...,279.72
2,CatBoost V5,C:\temp\shift_ml\models\final_ensemble_12_66\c...,40355.58
3,CatBoost V8,C:\temp\shift_ml\models\final_ensemble_12_66\c...,36641.77
4,CatBoost Stats,C:\temp\shift_ml\models\final_ensemble_12_66\c...,30388.46
5,CatBoost V10,C:\temp\shift_ml\models\final_ensemble_12_66\c...,29594.18
6,Retrieval reference,C:\temp\shift_ml\models\final_ensemble_12_66\r...,294.26
7,Target statistics reference,C:\temp\shift_ml\models\final_ensemble_12_66\t...,150.61
8,Model schemas,C:\temp\shift_ml\models\final_ensemble_12_66\m...,34.42
9,Ensemble config,C:\temp\shift_ml\models\final_ensemble_12_66\e...,0.73


## 13. Фиксация окружения

`requirements.txt` позволяет воспроизвести ключевые версии библиотек. Он создаётся в корне проекта и фиксирует текущее Python-окружение.

In [15]:
# ==============================================================
# 13. SAVE REQUIREMENTS
# ==============================================================

REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"

with open(
    REQUIREMENTS_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "freeze",
        ],
        stdout=file,
        check=True,
    )

print("Saved:", REQUIREMENTS_PATH)

Saved: C:\temp\shift_ml\requirements.txt


## Выводы Notebook 3

- На всей обучающей выборке обучены и сохранены все компоненты утверждённого ансамбля.
- Для моделей с target statistics сохранены reference-данные, необходимые для их расчёта на новых объектах.
- Для retrieval сохранена таблица аналогов с признаками и логарифмом цены.
- Для каждой модели сохранена точная схема признаков и список категориальных колонок.
- Веса ансамбля и conditional calibration не переобучались: используется конфигурация, выбранная по OOF/nested validation и подтверждённая leaderboard score `12.66`.
- Следующий шаг — отдельный inference notebook или скрипт, который загрузит эти артефакты и создаст `submission.csv`.

In [17]:
# ==============================================================
# 14. ADD AND SAVE V6 FOR DISAGREEMENT CALIBRATION
#
# V6 имеет нулевой вес в финальной сумме, но её прогноз участвует
# в расчёте relative_std всех компонентов ансамбля.
# Без V6 нельзя воспроизвести conditional calibration.
# ==============================================================

import json

from catboost import CatBoostRegressor


required_variables = [
    "X_train_cb",
    "X_test_cb",
    "train_canonical_aligned",
    "test_canonical_aligned",
    "V8_SUPPLEMENTARY_TITLE_FEATURES",
    "categorical_v8",
    "y_train",
    "CATBOOST_PARAMS",
    "FINAL_MODEL_DIR",
    "MODEL_SCHEMAS_PATH",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        "Не найдены переменные из Notebook 3: "
        f"{missing_variables}"
    )


# V6 соответствует базовому title-hierarchy набору признаков:
# без V8 supplementary title features, но с исходным Цветом.
V6_RAW_COLUMNS = ["Цвет"]

v6_base_columns = [
    column
    for column in X_train_cb.columns
    if column not in V8_SUPPLEMENTARY_TITLE_FEATURES
]

X_train_v6 = X_train_cb[
    v6_base_columns
].copy()

X_test_v6 = X_test_cb[
    v6_base_columns
].copy()

for column in V6_RAW_COLUMNS:
    if column not in train_canonical_aligned.columns:
        raise KeyError(
            f"В train canonical нет колонки: {column}"
        )

    if column not in test_canonical_aligned.columns:
        raise KeyError(
            f"В test canonical нет колонки: {column}"
        )

    X_train_v6[column] = (
        train_canonical_aligned[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_v6[column] = (
        test_canonical_aligned[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

categorical_v6 = [
    column
    for column in categorical_v8
    if column in v6_base_columns
] + V6_RAW_COLUMNS

X_train_v6 = X_train_v6[
    list(dict.fromkeys(X_train_v6.columns))
]

X_test_v6 = X_test_v6[
    X_train_v6.columns
]

assert X_train_v6.columns.tolist() == X_test_v6.columns.tolist()
assert len(X_train_v6) == len(y_train)

v6_final_model = CatBoostRegressor(
    **CATBOOST_PARAMS
)

v6_final_model.fit(
    X_train_v6,
    np.log1p(y_train),
    cat_features=categorical_v6,
)

V6_MODEL_PATH = (
    FINAL_MODEL_DIR
    / "catboost_v6_final.cbm"
)

v6_final_model.save_model(
    V6_MODEL_PATH
)

with open(
    MODEL_SCHEMAS_PATH,
    mode="r",
    encoding="utf-8",
) as file:
    model_feature_schemas = json.load(file)

model_feature_schemas["v6"] = {
    "feature_columns": X_train_v6.columns.tolist(),
    "categorical_columns": categorical_v6,
    "base_feature_columns": v6_base_columns,
    "raw_columns": V6_RAW_COLUMNS,
}

with open(
    MODEL_SCHEMAS_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        model_feature_schemas,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("Saved V6 model:", V6_MODEL_PATH)
print("Updated schemas:", MODEL_SCHEMAS_PATH)
print("V6 feature count:", X_train_v6.shape[1])

0:	learn: 0.6514509	total: 194ms	remaining: 9m 41s
500:	learn: 0.1554186	total: 55.3s	remaining: 4m 35s
1000:	learn: 0.1198140	total: 1m 54s	remaining: 3m 49s
1500:	learn: 0.0975160	total: 2m 57s	remaining: 2m 56s
2000:	learn: 0.0818566	total: 3m 55s	remaining: 1m 57s
2500:	learn: 0.0701413	total: 4m 59s	remaining: 59.8s
2999:	learn: 0.0608866	total: 6m 6s	remaining: 0us
Saved V6 model: C:\temp\shift_ml\models\final_ensemble_12_66\catboost_v6_final.cbm
Updated schemas: C:\temp\shift_ml\models\final_ensemble_12_66\model_feature_schemas.json
V6 feature count: 62


In [18]:
# ==============================================================
# 14. TRAIN AND SAVE V6 FOR CONDITIONAL CALIBRATION
#
# У V6 нулевой вес в итоговой сумме, но её прогноз участвует
# в расчёте disagreement всех компонентов ансамбля.
# Поэтому V6 нужна для воспроизводимого инференса.
# ==============================================================

import gc
import json

import numpy as np
from catboost import CatBoostRegressor


required_variables = [
    "X_train_cb",
    "X_test_cb",
    "train_canonical_aligned",
    "test_canonical_aligned",
    "categorical_v8",
    "y_train",
    "CATBOOST_PARAMS",
    "FINAL_MODEL_DIR",
    "MODEL_SCHEMAS_PATH",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        "В Notebook 3 не найдены переменные: "
        f"{missing_variables}"
    )


# В V6 не используются дополнительные V8-признаки комплектации.
# Но используется исходный Цвет — это отличает V6 от V7.
V8_SUPPLEMENTARY_TITLE_FEATURES = [
    "Название_комплектация_1",
    "Название_комплектация_2",
    "Название_комплектация_3",
    "Название_первые_2",
    "Название_первые_3",
    "Название_число_токенов",
    "Название_флаг_QUATTRO",
    "Название_флаг_XDRIVE",
    "Название_флаг_4MATIC",
    "Название_флаг_ALLGRIP",
    "Название_флаг_ALLTRACK",
    "Название_флаг_EQUIPPED",
]

V6_RAW_COLUMNS = [
    "Цвет",
]

v6_base_columns = [
    column
    for column in X_train_cb.columns
    if column not in V8_SUPPLEMENTARY_TITLE_FEATURES
]

X_train_v6 = X_train_cb[
    v6_base_columns
].copy()

X_test_v6 = X_test_cb[
    v6_base_columns
].copy()

for column in V6_RAW_COLUMNS:
    if column not in train_canonical_aligned.columns:
        raise KeyError(
            f"В train canonical нет колонки: {column}"
        )

    if column not in test_canonical_aligned.columns:
        raise KeyError(
            f"В test canonical нет колонки: {column}"
        )

    X_train_v6[column] = (
        train_canonical_aligned[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_v6[column] = (
        test_canonical_aligned[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

categorical_v6 = list(
    dict.fromkeys(
        [
            column
            for column in categorical_v8
            if column in X_train_v6.columns
        ]
        + V6_RAW_COLUMNS
    )
)

for column in categorical_v6:
    X_train_v6[column] = (
        X_train_v6[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_v6[column] = (
        X_test_v6[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

assert X_train_v6.columns.tolist() == X_test_v6.columns.tolist()
assert len(X_train_v6) == len(y_train)

v6_final_model = CatBoostRegressor(
    **CATBOOST_PARAMS
)

v6_final_model.fit(
    X_train_v6,
    np.log1p(y_train),
    cat_features=categorical_v6,
)

V6_MODEL_PATH = (
    FINAL_MODEL_DIR
    / "catboost_v6_final.cbm"
)

v6_final_model.save_model(
    V6_MODEL_PATH
)

with open(
    MODEL_SCHEMAS_PATH,
    mode="r",
    encoding="utf-8",
) as file:
    model_feature_schemas = json.load(file)

model_feature_schemas["v6"] = {
    "feature_columns": X_train_v6.columns.tolist(),
    "categorical_columns": categorical_v6,
    "base_feature_columns": v6_base_columns,
    "raw_columns": V6_RAW_COLUMNS,
}

with open(
    MODEL_SCHEMAS_PATH,
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        model_feature_schemas,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("Saved V6 model:", V6_MODEL_PATH)
print("Updated schema:", MODEL_SCHEMAS_PATH)
print("V6 feature count:", X_train_v6.shape[1])

gc.collect()

0:	learn: 0.6514509	total: 135ms	remaining: 6m 43s
500:	learn: 0.1554186	total: 1m 2s	remaining: 5m 13s
1000:	learn: 0.1198140	total: 1m 55s	remaining: 3m 50s
1500:	learn: 0.0975160	total: 2m 49s	remaining: 2m 49s
2000:	learn: 0.0818566	total: 3m 44s	remaining: 1m 52s
2500:	learn: 0.0701413	total: 4m 43s	remaining: 56.5s
2999:	learn: 0.0608866	total: 5m 39s	remaining: 0us
Saved V6 model: C:\temp\shift_ml\models\final_ensemble_12_66\catboost_v6_final.cbm
Updated schema: C:\temp\shift_ml\models\final_ensemble_12_66\model_feature_schemas.json
V6 feature count: 62


988